# CDK4 Protein-Ligand Affinity Prediction with MSA

This notebook demonstrates:
1. MSA search for CDK4 protein using GPU MSA NIM
2. Converting MSA from JSON to a3m format
3. Protein-ligand affinity prediction using Boltz2 NIM

**Target**: CDK4 (Cyclin-dependent kinase 4)  
**Ligand**: Palbociclib (FDA-approved CDK4/6 inhibitor)

## 1. Setup and Imports

In [1]:
import asyncio
import json
import os
from pathlib import Path
from datetime import datetime

from boltz2_client import Boltz2Client, Polymer, Ligand, PredictionRequest
from boltz2_client.models import AlignmentFileRecord
from boltz2_client.msa_search import MSASearchClient

# Endpoints
MSA_ENDPOINT = os.getenv("MSA_NIM_URL", "http://localhost:8000")
BOLTZ2_ENDPOINT = os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000")

print("✅ Imports completed")
print(f"MSA Endpoint: {MSA_ENDPOINT}")
print(f"Boltz2 Endpoint: {BOLTZ2_ENDPOINT}")

✅ Imports completed
MSA Endpoint: http://localhost:8000
Boltz2 Endpoint: http://localhost:8000


## 2. Define CDK4 and Palbociclib

In [2]:
# CDK4 sequence (295 amino acids)
CDK4_SEQUENCE = "MATSRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGGGGGLPISTVREVALLRRLEAFEHPNVVRLMDVCATSRTDREIKVTLVFEHVDQDLRTYLDKAPPPGLPAETIKDLMRQFLRGLDFLHANCIVHRDLKPENILVTSGGTVKLADFGLARIYSYQMALTPVVVTLWYRAPEVLLQSTYATPVDMWSVGCIFAEMFRRKPLFCGNSEADQLGKIFDLIGLPPEDDWPRDVSLPRGAFPPRGPRPVQSVVPEMEESGAQLLLEMLTFNPHKRISAFRALQHSYLHKDEGNPE"

# Palbociclib (Ibrance) - CDK4/6 inhibitor
# SMILES from PubChem CID: 5330286
PALBOCICLIB_SMILES = "CC1=C(C(=O)N(C2=NC(=NC=C12)NC3=NC=C(C=C3)N4CCNCC4)C5CCCC5)C(=O)C"

print(f"🧬 CDK4 Protein:")
print(f"   Length: {len(CDK4_SEQUENCE)} residues")
print(f"   Function: Cell cycle regulation (G1/S transition)")
print(f"\n💊 Palbociclib (Ibrance):")
print(f"   Type: CDK4/6 selective inhibitor")
print(f"   FDA approved: 2015 (breast cancer treatment)")
print(f"   Known IC50: ~11 nM for CDK4")

🧬 CDK4 Protein:
   Length: 303 residues
   Function: Cell cycle regulation (G1/S transition)

💊 Palbociclib (Ibrance):
   Type: CDK4/6 selective inhibitor
   FDA approved: 2015 (breast cancer treatment)
   Known IC50: ~11 nM for CDK4


### Important Note: MSA Search Parameters

The MSA search might return only the query sequence if:
1. The parameter name is wrong (`max_results` → `max_msa_sequences`)
2. The E-value is too strict (default 0.0001)
3. The protein sequence has few homologs in the database

The cell below uses the correct parameters.


## 3. MSA Search for CDK4

In [3]:
# Initialize MSA client
msa_client = MSASearchClient(endpoint_url=MSA_ENDPOINT)

# Create output directory
output_dir = Path("cdk4_msa_affinity")
output_dir.mkdir(exist_ok=True)

print("🔍 Searching for CDK4 MSA...")

# Perform MSA search with CORRECT parameters
msa_response = await msa_client.search(
    sequence=CDK4_SEQUENCE,
    databases=["Uniref30_2302", "colabfold_envdb_202108"],
    max_msa_sequences=500,  # Correct parameter name (not max_results)
    e_value=0.1,  # More permissive than default 0.0001
    output_alignment_formats=["a3m"]
)

# client = Boltz2Client()
# client.configure_msa_search(msa_endpoint_url=MSA_ENDPOINT)

# # Then search using the client's MSA methods
# msa_response = await client.search_msa(
#     sequence=CDK4_SEQUENCE,
#     databases=["Uniref30_2302"] #, "colabfold_envdb_202108"],
#     max_msa_sequences=500
# )


print(f"\n✅ MSA search complete!")
print(f"Found alignments in {len(msa_response.alignments)} databases:")
for db_name, alignments in msa_response.alignments.items():
    print(f"   - {db_name}: {len(alignments)} sequences")

🔍 Searching for CDK4 MSA...

✅ MSA search complete!
Found alignments in 3 databases:
   - Uniref30_2302: 1 sequences
   - colabfold_envdb_202108: 1 sequences
   - colabfold: 1 sequences


In [4]:
msa_response

MSASearchResponse(alignments={'Uniref30_2302': {'a3m': AlignmentFileRecord(alignment='>Query|-|Query\nMATSRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGGGGGLPISTVREVALLRRLEAFEHPNVVRLMDVCATSRTDREIKVTLVFEHVDQDLRTYLDKAPPPGLPAETIKDLMRQFLRGLDFLHANCIVHRDLKPENILVTSGGTVKLADFGLARIYSYQMALTPVVVTLWYRAPEVLLQSTYATPVDMWSVGCIFAEMFRRKPLFCGNSEADQLGKIFDLIGLPPEDDWPRDVSLPRGAFPPRGPRPVQSVVPEMEESGAQLLLEMLTFNPHKRISAFRALQHSYLHKDEGNPE\n>UniRef100_A0A8D2D641\t422\t0.989\t1.627E-129\t0\t298\t303\t0\t298\t302\nMATSRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGAGGGLPISTVREVALLRRLEAFEHPNVVRLMDVCATSRTDREIKVTLVFEHVDQDLRTYLDKAPPPGLPTETIKDLMRQFLRGLDFLHANCIVHRDLKPENILVTSGGTVKLADFGLARIYSYQMALTPVVVTLWYRAPEVLLQSTYATPVDMWSVGCIFAEMFRRKPLFCGNSEADQLGKIFDLIGLPPEDDWPRDVSLPRGAFPPRGPRPVQSVVPEMEESGAQLLLEMLTFNPHKRISAFRALQHSYLHKEE----\n>UniRef100_UPI0003BC0225\t421\t0.947\t7.856E-129\t0\t302\t303\t0\t302\t303\nMATPRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGAGGGLPVSTVREVALLRRLEAFEHPNVVRLMDVCATARTDRETKVTLVFEHVDQDLRMYLDKAPPPGLPVETIKDLMRQLLRGLDFLHANCIVHR

## 4. Convert MSA from JSON to A3M Format

In [5]:
def convert_msa_to_a3m(msa_response, query_sequence, query_name="query"):
    """
    Convert MSA search response to A3M format.
    A3M format is like FASTA but allows lowercase letters for insertions.
    """
    a3m_lines = []
    
    # Add query sequence first
    a3m_lines.append(f">{query_name}")
    a3m_lines.append(query_sequence)
    
    # Process each database's alignments
    sequence_count = 0
    for db_name, alignments in msa_response.alignments.items():
        print(f"\nProcessing {db_name} alignments...")
        
        for i, alignment in enumerate(alignments):
            # Extract aligned sequence
            if hasattr(alignment, 'aligned_sequence'):
                aligned_seq = alignment.aligned_sequence
            elif hasattr(alignment, 'sequence'):
                aligned_seq = alignment.sequence
            else:
                continue
            
            # Skip if identical to query
            if aligned_seq.replace('-', '') == query_sequence:
                continue
            
            # Create header with metadata
            header = f">{db_name}_{i}"
            if hasattr(alignment, 'description'):
                header += f" {alignment.description[:50]}"
            if hasattr(alignment, 'e_value'):
                header += f" E={alignment.e_value:.2e}"
            
            a3m_lines.append(header)
            a3m_lines.append(aligned_seq)
            sequence_count += 1
    
    print(f"\n✅ Converted {sequence_count} sequences to A3M format")
    return "\n".join(a3m_lines)

# Convert MSA to A3M
cdk4_a3m = convert_msa_to_a3m(msa_response, CDK4_SEQUENCE, "CDK4_HUMAN")

# Save A3M file
a3m_path = output_dir / "cdk4_msa.a3m"
with open(a3m_path, "w") as f:
    f.write(cdk4_a3m)

print(f"\n📄 A3M file saved to: {a3m_path}")
print(f"File size: {len(cdk4_a3m):,} bytes")
print(f"\nFirst few lines of A3M:")
print("\n".join(cdk4_a3m.split("\n")[:6]))
print("...")


Processing Uniref30_2302 alignments...

Processing colabfold_envdb_202108 alignments...

Processing colabfold alignments...

✅ Converted 0 sequences to A3M format

📄 A3M file saved to: cdk4_msa_affinity/cdk4_msa.a3m
File size: 315 bytes

First few lines of A3M:
>CDK4_HUMAN
MATSRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGGGGGLPISTVREVALLRRLEAFEHPNVVRLMDVCATSRTDREIKVTLVFEHVDQDLRTYLDKAPPPGLPAETIKDLMRQFLRGLDFLHANCIVHRDLKPENILVTSGGTVKLADFGLARIYSYQMALTPVVVTLWYRAPEVLLQSTYATPVDMWSVGCIFAEMFRRKPLFCGNSEADQLGKIFDLIGLPPEDDWPRDVSLPRGAFPPRGPRPVQSVVPEMEESGAQLLLEMLTFNPHKRISAFRALQHSYLHKDEGNPE
...


## 5. Alternative: Direct MSA Format Conversion

There are two ways to use MSA search integration:

### Option 1: Using MSASearchIntegration directly (shown below)
### Option 2: Using Boltz2Client (simpler)

In [6]:
# Alternative approach using MSASearchIntegration
from boltz2_client.msa_search import MSASearchIntegration

# First create MSA client
msa_client_for_integration = MSASearchClient(endpoint_url=MSA_ENDPOINT)

# Then create integration with the client
msa_integration = MSASearchIntegration(msa_client_for_integration)

# Search and save directly in A3M format
msa_file_path = await msa_integration.search_and_save(
    sequence=CDK4_SEQUENCE,
    output_path=output_dir / "cdk4_direct.a3m",
    output_format="a3m",
    databases=["Uniref30_2302", "colabfold_envdb_202108"],
    max_msa_sequences=500
)

print(f"✅ Direct MSA saved to: {msa_file_path}")

✅ Direct MSA saved to: cdk4_msa_affinity/cdk4_direct.a3m


### Option 2: Using Boltz2Client (Simpler Approach)


In [7]:
# Simpler approach using Boltz2Client
client_with_msa = Boltz2Client()

# Configure MSA search
client_with_msa.configure_msa_search(msa_endpoint_url=MSA_ENDPOINT)

# Search and save MSA directly
msa_file_simple = await client_with_msa.search_msa(
    sequence=CDK4_SEQUENCE,
    databases=["Uniref30_2302", "colabfold_envdb_202108"],
    max_msa_sequences=500,
    output_format="a3m",
    save_path=output_dir / "cdk4_simple.a3m"
)

print(f"✅ MSA saved using Boltz2Client: {msa_file_simple}")


✅ MSA Search configured: http://localhost:8000

✅ MSA saved using Boltz2Client: cdk4_msa_affinity/cdk4_simple.a3m


## 6. Prepare Protein with MSA for Boltz2

In [18]:
# Load the A3M content
with open(msa_file_simple, "r") as f:
    a3m_content = f.read()

# Create CDK4 polymer with MSA
cdk4_with_msa = Polymer(
    id="A",
    molecule_type="protein",
    sequence=CDK4_SEQUENCE,
    msa={
        "default": {
            "a3m": AlignmentFileRecord(
                alignment=a3m_content,
                format="a3m",
                rank=0
            )
        }
    }
)

print("✅ CDK4 polymer created with MSA")
print(f"   MSA sequences: {len(a3m_content.split('>')) - 1}")

✅ CDK4 polymer created with MSA
   MSA sequences: 101


## Troubleshooting: Server Error

If you encounter a "DataLoader worker exited unexpectedly" error, try these steps:


In [19]:
# Step 1: Test basic server connectivity
print("🔍 Testing Boltz2 server connectivity...")

try:
    # Simple test with small protein
    test_client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)
    test_polymer = Polymer(
        id="A",
        molecule_type="protein",
        sequence="MKTVRQERLKSIVRILERSKEPVSGAQ"  # Short 28-residue sequence
    )
    test_request = PredictionRequest(
        polymers=[test_polymer],
        recycling_steps=1,
        sampling_steps=10
    )
    
    print("Testing basic prediction...")
    test_result = await test_client.predict(test_request)
    print("✅ Server is responding to basic requests")
    
except Exception as e:
    print(f"❌ Server connectivity issue: {e}")
    print("\nPossible solutions:")
    print("1. Check if Boltz2 NIM is running: http://localhost:8000")
    print("2. Update BOLTZ2_ENDPOINT variable with correct URL")
    print("3. Check server logs for errors")


🔍 Testing Boltz2 server connectivity...
Testing basic prediction...
✅ Server is responding to basic requests


In [20]:
# Step 2: Try CDK4 without MSA or affinity
print("\n🧪 Testing CDK4 prediction without MSA or affinity...")

# Initialize client if not already done
if 'client' not in locals():
    from boltz2_client import Boltz2Client
    client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)
    print("✅ Client initialized")

try:
    # CDK4 without MSA
    cdk4_simple = Polymer(
        id="A",
        molecule_type="protein",
        sequence=CDK4_SEQUENCE
        # No MSA
    )
    
    # Palbociclib without affinity prediction
    palbociclib_simple = Ligand(
        id="LIG",
        smiles=PALBOCICLIB_SMILES,
        predict_affinity=False  # Disabled
    )
    
    simple_request = PredictionRequest(
        polymers=[cdk4_simple],
        ligands=[palbociclib_simple],
        recycling_steps=3,
        sampling_steps=50
        # No affinity parameters
    )
    
    print("Predicting without MSA or affinity...")
    simple_result = await client.predict(simple_request)
    
    # Save simple result
    with open(output_dir / "cdk4_palbociclib_simple.cif", "w") as f:
        f.write(simple_result.structures[0].structure)
    
    print("✅ Basic CDK4-Palbociclib prediction works!")
    print(f"   pTM: {simple_result.ptm_scores[0]:.3f}")
    print(f"   pLDDT: {simple_result.complex_plddt_scores[0]:.1f}")
    
except Exception as e:
    print(f"❌ Error with basic prediction: {e}")
    print("\nThe server may be overloaded or misconfigured.")



🧪 Testing CDK4 prediction without MSA or affinity...
Predicting without MSA or affinity...
✅ Basic CDK4-Palbociclib prediction works!
   pTM: 0.944
   pLDDT: 0.9


In [11]:
# Step 3: Try with MSA but without affinity
print("\n🧬 Testing CDK4 with MSA but no affinity...")

# Initialize client if not already done
if 'client' not in locals():
    from boltz2_client import Boltz2Client
    client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)
    print("✅ Client initialized")

try:
    # Use the MSA we already loaded
    cdk4_with_msa_only = cdk4_with_msa  # From earlier cell
    
    # Ligand without affinity
    palbociclib_no_affinity = Ligand(
        id="LIG",
        smiles=PALBOCICLIB_SMILES,
        predict_affinity=False  # No affinity
    )
    
    msa_only_request = PredictionRequest(
        polymers=[cdk4_with_msa_only],
        ligands=[palbociclib_no_affinity],
        recycling_steps=3,
        sampling_steps=50
    )
    
    print("Predicting with MSA but no affinity...")
    msa_result = await client.predict(msa_only_request)
    
    print("✅ MSA-enhanced prediction works!")
    print(f"   pTM: {msa_result.ptm_scores[0]:.3f}")
    print(f"   pLDDT: {msa_result.complex_plddt_scores[0]:.1f}")
    
    # Check if MSA improved accuracy
    if 'simple_result' in locals():
        ptm_improvement = msa_result.ptm_scores[0] - simple_result.ptm_scores[0]
        print(f"\n📈 MSA improvement: {ptm_improvement:+.3f} pTM")
    
except Exception as e:
    print(f"❌ Error with MSA prediction: {e}")
    print("\nThe MSA might be too large or the server has memory constraints.")



🧬 Testing CDK4 with MSA but no affinity...
Predicting with MSA but no affinity...
✅ MSA-enhanced prediction works!
   pTM: 0.943
   pLDDT: 0.9

📈 MSA improvement: +0.011 pTM


In [21]:
# Step 4: Try affinity without MSA
print("\n💊 Testing affinity prediction without MSA...")

# Initialize client if not already done
if 'client' not in locals():
    from boltz2_client import Boltz2Client
    client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)
    print("✅ Client initialized")

try:
    # CDK4 without MSA
    cdk4_no_msa = Polymer(
        id="A",
        molecule_type="protein",
        sequence=CDK4_SEQUENCE
    )
    
    # Ligand WITH affinity
    palbociclib_affinity = Ligand(
        id="LIG",
        smiles=PALBOCICLIB_SMILES,
        predict_affinity=True  # Enable affinity
    )
    
    affinity_only_request = PredictionRequest(
        polymers=[cdk4_no_msa],
        ligands=[palbociclib_affinity],
        recycling_steps=3,
        sampling_steps=50,
        # Reduced affinity parameters
        sampling_steps_affinity=100,  # Reduced from 200
        diffusion_samples_affinity=3,  # Reduced from 5
        affinity_mw_correction=True
    )
    
    print("Predicting with affinity but no MSA...")
    affinity_result = await client.predict(affinity_only_request)
    
    print("✅ Affinity prediction works!")
    
    if affinity_result.affinities and "LIG" in affinity_result.affinities:
        aff = affinity_result.affinities["LIG"]
        ic50_nm = (10 ** (-aff.affinity_pic50[0])) * 1e9
        print(f"   pIC50: {aff.affinity_pic50[0]:.2f}")
        print(f"   IC50: {ic50_nm:.1f} nM")
        print(f"   Binding probability: {aff.affinity_probability_binary[0]:.1%}")
    
except Exception as e:
    print(f"❌ Error with affinity prediction: {e}")
    print("\nAffinity prediction may require more resources.")
    print("Try reducing sampling_steps_affinity and diffusion_samples_affinity.")



💊 Testing affinity prediction without MSA...
Predicting with affinity but no MSA...
✅ Affinity prediction works!
   pIC50: 8.86
   IC50: 1.4 nM
   Binding probability: 40.5%


## Recommended Approach Based on Server Capabilities

Based on the troubleshooting results above, use the approach that works for your server:


In [22]:
# Final optimized prediction with reduced parameters
print("🎯 Running optimized CDK4-Palbociclib prediction...")

# Initialize client if not already done
if 'client' not in locals():
    from boltz2_client import Boltz2Client
    client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)
    print("✅ Client initialized")

# Define palbociclib ligand if not already done
if 'palbociclib' not in locals():
    from boltz2_client.models import Ligand
    palbociclib = Ligand(
        id="LIG",
        smiles=PALBOCICLIB_SMILES,
        predict_affinity=True
    )
    print("✅ Palbociclib ligand defined")

# Ensure cdk4_with_msa is defined
if 'cdk4_with_msa' not in locals():
    print("⚠️ Warning: cdk4_with_msa not found, using CDK4 without MSA")
    from boltz2_client.models import Polymer
    cdk4_with_msa = Polymer(
        id="A",
        molecule_type="protein",
        sequence=CDK4_SEQUENCE
    )

# Choose the configuration that worked in troubleshooting
try:
    # Reduced complexity request
    optimized_request = PredictionRequest(
        polymers=[cdk4_with_msa],  # Use MSA if it worked
        ligands=[palbociclib],     # With affinity if server supports it
        recycling_steps=3,         # Standard
        sampling_steps=50,         # Standard
        # Reduced affinity parameters for stability
        sampling_steps_affinity=100,    # Reduced from 200
        diffusion_samples_affinity=3,   # Reduced from 5
        affinity_mw_correction=True
    )
    
    print("Parameters:")
    print(f"  MSA: {'Yes' if cdk4_with_msa.msa else 'No'}")
    print(f"  Affinity: {'Yes' if palbociclib.predict_affinity else 'No'}")
    print(f"  Sampling steps (affinity): 100")
    print(f"  Diffusion samples (affinity): 3")
    
    final_result = await client.predict(optimized_request)
    
    # Save final structure
    final_path = output_dir / "cdk4_palbociclib_optimized.cif"
    with open(final_path, "w") as f:
        f.write(final_result.structures[0].structure)
    
    print(f"\n✅ Success! Structure saved to: {final_path}")
    
    # Display results
    print(f"\n📊 Final Results:")
    print(f"├─ pTM: {final_result.ptm_scores[0]:.3f}")
    print(f"└─ Complex pLDDT: {final_result.complex_plddt_scores[0]:.1f}")
    
    if final_result.affinities and "LIG" in final_result.affinities:
        aff = final_result.affinities["LIG"]
        ic50_nm = (10 ** (-aff.affinity_pic50[0])) * 1e9
        print(f"\n💊 Affinity Prediction:")
        print(f"├─ pIC50: {aff.affinity_pic50[0]:.2f}")
        print(f"├─ IC50: {ic50_nm:.1f} nM (Known: 11 nM)")
        print(f"└─ Fold difference: {ic50_nm/11:.1f}x")
        
except Exception as e:
    print(f"❌ Still encountering errors: {e}")
    print("\nRecommendations:")
    print("1. Check server GPU memory: nvidia-smi")
    print("2. Restart Boltz2 NIM service")
    print("3. Use smaller proteins for testing")
    print("4. Contact server administrator")


🎯 Running optimized CDK4-Palbociclib prediction...
Parameters:
  MSA: Yes
  Affinity: Yes
  Sampling steps (affinity): 100
  Diffusion samples (affinity): 3

✅ Success! Structure saved to: cdk4_msa_affinity/cdk4_palbociclib_optimized.cif

📊 Final Results:
├─ pTM: 0.938
└─ Complex pLDDT: 0.9

💊 Affinity Prediction:
├─ pIC50: 8.52
├─ IC50: 3.0 nM (Known: 11 nM)
└─ Fold difference: 0.3x


## 7. Protein-Ligand Affinity Prediction (Updated with Reduced Parameters)

Due to the server error, here's an updated version with reduced parameters:


## Troubleshooting MSA Search

If your MSA search only returns 1 sequence (the query):

1. **Check parameters**: Use `max_msa_sequences` not `max_results`
2. **Adjust E-value**: Try 0.1 or 1.0 instead of default 0.0001
3. **Try different databases**: Some proteins are better represented in certain databases
4. **Use conserved domains**: Full sequences might be too specific

Test scripts available:
- `test_cdk4_msa_fixed.py` - Comprehensive MSA troubleshooting
- `quick_msa_test.py` - Quick parameter test
- `msa_issue_summary.md` - Complete issue analysis

Even without homologs, Boltz2 can still make predictions using the single sequence.


In [14]:
# Initialize Boltz2 client (if not already done)
if 'client' not in locals():
    client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)

# Create Palbociclib ligand with affinity prediction enabled
palbociclib = Ligand(
    id="LIG",
    smiles=PALBOCICLIB_SMILES,
    predict_affinity=True  # Enable affinity prediction
)

# Create prediction request with REDUCED parameters
request = PredictionRequest(
    polymers=[cdk4_with_msa],
    ligands=[palbociclib],
    recycling_steps=3,       # Reduced from 5
    sampling_steps=50,       # Reduced from 100
    # Affinity prediction parameters (significantly reduced)
    sampling_steps_affinity=100,    # Reduced from 200
    diffusion_samples_affinity=3,   # Reduced from 5
    affinity_mw_correction=True
)

print("🔄 Predicting CDK4-Palbociclib complex with affinity...")
print("   Using reduced parameters for server stability")
print(f"   Protein: {len(CDK4_SEQUENCE)} residues (with MSA)")
print(f"   Ligand: Palbociclib")
print(f"   Affinity: sampling_steps=100, diffusion_samples=3")

try:
    # Predict
    result = await client.predict(request)
    
    # Save structure
    structure_path = output_dir / "cdk4_palbociclib_complex.cif"
    with open(structure_path, "w") as f:
        f.write(result.structures[0].structure)
    
    print(f"\n✅ Prediction complete!")
    print(f"📁 Structure saved to: {structure_path}")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nTry running the troubleshooting cells above to identify the issue.")
    print("Or use the standalone test script: python examples/test_boltz2_server.py")


🔄 Predicting CDK4-Palbociclib complex with affinity...
   Using reduced parameters for server stability
   Protein: 303 residues (with MSA)
   Ligand: Palbociclib
   Affinity: sampling_steps=100, diffusion_samples=3

✅ Prediction complete!
📁 Structure saved to: cdk4_msa_affinity/cdk4_palbociclib_complex.cif


## 7. Protein-Ligand Affinity Prediction

In [15]:
# Initialize Boltz2 client
client = Boltz2Client(base_url=BOLTZ2_ENDPOINT)

# Create Palbociclib ligand with affinity prediction enabled
palbociclib = Ligand(
    id="LIG",
    smiles=PALBOCICLIB_SMILES,
    predict_affinity=True  # Enable affinity prediction
)

# Create prediction request
request = PredictionRequest(
    polymers=[cdk4_with_msa],
    ligands=[palbociclib],
    recycling_steps=5,
    sampling_steps=100,
    # Affinity prediction parameters
    sampling_steps_affinity=200,
    diffusion_samples_affinity=5,
    affinity_mw_correction=True
)

print("🔄 Predicting CDK4-Palbociclib complex with affinity...")
print("   This may take several minutes...")

# Predict
result = await client.predict(request)

# Save structure
structure_path = output_dir / "cdk4_palbociclib_complex.cif"
with open(structure_path, "w") as f:
    f.write(result.structures[0].structure)

print(f"\n✅ Prediction complete!")
print(f"📁 Structure saved to: {structure_path}")

🔄 Predicting CDK4-Palbociclib complex with affinity...
   This may take several minutes...

✅ Prediction complete!
📁 Structure saved to: cdk4_msa_affinity/cdk4_palbociclib_complex.cif


## 8. Analyze Results

In [16]:
# Structure confidence
print("📊 Structure Confidence Metrics:")
print(f"├─ Overall confidence: {result.confidence_scores[0]:.1%}")
print(f"├─ pTM: {result.ptm_scores[0]:.3f}")
print(f"└─ Complex pLDDT: {result.complex_plddt_scores[0]:.1f}")

# Affinity predictions
if result.affinities and "LIG" in result.affinities:
    affinity = result.affinities["LIG"]
    
    print("\n💊 Affinity Predictions:")
    print(f"├─ pIC50: {affinity.affinity_pic50[0]:.2f}")
    
    # Convert pIC50 to IC50 in nM
    ic50_molar = 10 ** (-affinity.affinity_pic50[0])
    ic50_nm = ic50_molar * 1e9
    print(f"├─ IC50: {ic50_nm:.1f} nM")
    print(f"├─ Binding probability: {affinity.affinity_probability_binary[0]:.1%}")
    
    # Compare with known value
    known_ic50 = 11  # nM
    print(f"\n📈 Comparison with experimental data:")
    print(f"├─ Predicted IC50: {ic50_nm:.1f} nM")
    print(f"├─ Known IC50: {known_ic50} nM")
    print(f"└─ Fold difference: {ic50_nm/known_ic50:.1f}x")
    
    if 0.2 <= ic50_nm/known_ic50 <= 5:
        print("\n✅ Excellent prediction! Within 5-fold of experimental value.")
    elif 0.1 <= ic50_nm/known_ic50 <= 10:
        print("\n✅ Good prediction! Within 10-fold of experimental value.")
    else:
        print("\n⚠️  Prediction differs significantly from experimental value.")
else:
    print("\n❌ No affinity predictions found")

📊 Structure Confidence Metrics:
├─ Overall confidence: 91.5%
├─ pTM: 0.930
└─ Complex pLDDT: 0.9

💊 Affinity Predictions:
├─ pIC50: 8.88
├─ IC50: 1.3 nM
├─ Binding probability: 43.5%

📈 Comparison with experimental data:
├─ Predicted IC50: 1.3 nM
├─ Known IC50: 11 nM
└─ Fold difference: 0.1x

✅ Good prediction! Within 10-fold of experimental value.


## 9. Save Complete Results

In [17]:
# Save all results to JSON
results_data = {
    "timestamp": datetime.now().isoformat(),
    "protein": {
        "name": "CDK4",
        "sequence_length": len(CDK4_SEQUENCE),
        "msa_sequences": len(a3m_content.split('>')) - 1
    },
    "ligand": {
        "name": "Palbociclib",
        "smiles": PALBOCICLIB_SMILES,
        "known_ic50_nm": 11
    },
    "structure_confidence": {
        "overall": result.confidence_scores[0],
        "ptm": result.ptm_scores[0],
        "plddt": result.complex_plddt_scores[0]
    }
}

if result.affinities and "LIG" in result.affinities:
    affinity = result.affinities["LIG"]
    results_data["affinity_predictions"] = {
        "pic50": affinity.affinity_pic50[0],
        "ic50_nm": ic50_nm,
        "binding_probability": affinity.affinity_probability_binary[0]
    }

# Save JSON
json_path = output_dir / "cdk4_palbociclib_results.json"
with open(json_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f"\n📄 Complete results saved to: {json_path}")

# Summary of all outputs
print("\n📁 All output files:")
print(f"├─ {a3m_path.name} - MSA in A3M format")
print(f"├─ {structure_path.name} - Complex structure")
print(f"└─ {json_path.name} - Complete results")


📄 Complete results saved to: cdk4_msa_affinity/cdk4_palbociclib_results.json

📁 All output files:
├─ cdk4_msa.a3m - MSA in A3M format
├─ cdk4_palbociclib_complex.cif - Complex structure
└─ cdk4_palbociclib_results.json - Complete results


## 10. Visualization Commands

To visualize the results in PyMOL:

```bash
# Open structure
pymol cdk4_msa_affinity/cdk4_palbociclib_complex.cif

# Color by confidence (B-factor)
spectrum b, red_yellow_green_blue, minimum=50, maximum=90

# Show ligand as sticks
show sticks, resn LIG
color magenta, resn LIG

# Show binding site
show sticks, byres resn LIG around 5
```

## Key Takeaways

1. **MSA improves accuracy**: Using MSA typically enhances both structure and affinity predictions
2. **A3M format**: Standard format for MSAs in structure prediction
3. **Affinity validation**: Compare predictions with known experimental values
4. **CDK4-Palbociclib**: Well-validated system for testing predictions

This workflow can be adapted for any protein-ligand system where you want MSA-enhanced affinity predictions!